# Instruction
Copy this notebook to the root of the server with the jupyter notebooks to download all the folders and subfolders with their files.  
On a Jupyter server, run `pip install ipynbname` once so this notebook can exclude itself from the zip by name; otherwise it still excludes itself by content.

In [ ]:
# Cell 1: Create a zip of all files in the current directory and subfolders
import os
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

# Root to zip (current directory when notebook runs)
root = Path(".")
zip_path = Path("all_notebooks_and_files.zip")

# Exclude the running notebook (this file) regardless of its name
this_notebook_path = None
try:
    import ipynbname
    this_notebook_path = Path(ipynbname.path()).resolve()
except Exception:
    try:
        this_notebook_path = Path(__vsc_ipynb_file__).resolve()
    except NameError:
        pass

# Optional: skip the zip file itself and hidden dirs
skip_dirs = {".git", "__pycache__", ".ipynb_checkpoints", ".cursor"}
skip_names = {zip_path.name}

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in root.rglob("*"):
        if not f.is_file():
            continue
        if any(part in skip_dirs for part in f.parts):
            continue
        if f.name in skip_names:
            continue
        # Don't include this notebook (the runner) when path is known (server or VS Code)
        if this_notebook_path is not None and f.resolve() == this_notebook_path:
            continue
        # Fallback: exclude any .ipynb that contains our zip filename (identifies this script)
        if f.suffix == ".ipynb":
            try:
                if zip_path.name in f.read_text():
                    continue
            except Exception:
                pass
        zf.write(f, f.relative_to(root))

print(f"Created {zip_path} with all subfolders and files.")
display(FileLink(zip_path, result_html_prefix="Download: "))